<a href="https://colab.research.google.com/github/ryouchinsa/sam-cpp-macos/blob/master/sam_cpp_macos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Segment Anything Model 2 CPP Wrapper for macOS and Ubuntu GPU

This code is to run a Segment Anything Model 2 ONNX model in c++ code and implemented on the macOS app RectLabel.

### Use GPU

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Runtime` -> `Change runtime type` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [51]:
!nvidia-smi

Tue Mar  3 18:03:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install packages

In [53]:
!sudo apt-get update
!sudo apt-get install build-essential tar curl zip unzip autopoint libtool bison libx11-dev libxft-dev libxext-dev libxrandr-dev libxi-dev libxcursor-dev libxdamage-dev libxinerama-dev libxtst-dev cmake libgflags-dev libopencv-dev python3-dev

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

### Download ONNX Runtime

In [56]:
!wget https://github.com/microsoft/onnxruntime/releases/download/v1.23.2/onnxruntime-linux-x64-gpu-1.23.2.tgz
!tar -xzvpf onnxruntime-linux-x64-gpu-1.23.2.tgz

--2026-03-03 18:04:48--  https://github.com/microsoft/onnxruntime/releases/download/v1.23.2/onnxruntime-linux-x64-gpu-1.23.2.tgz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/156939672/344eed64-4968-451e-82e5-442025aa1c43?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-03T19%3A02%3A01Z&rscd=attachment%3B+filename%3Donnxruntime-linux-x64-gpu-1.23.2.tgz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-03T18%3A01%3A50Z&ske=2026-03-03T19%3A02%3A01Z&sks=b&skv=2018-11-09&sig=2B%2BbJg6i3x95HcdKfHdfkA7ENoA%2F4QKDkw1Ah9u3mqw%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjU2NDY4OCwibmJmIjoxNzcyN

### Install SAM2

Install Segment Anything Model 2, download checkpoints and copy yaml files in sam2/configs/sam2.1 to sam2.

In [78]:
!git clone https://github.com/facebookresearch/sam2.git

Cloning into 'sam2'...
remote: Enumerating objects: 1070, done.
remote: Total 1070 (delta 0), reused 0 (delta 0), pack-reused 1070 (from 1)
Receiving objects: 100% (1070/1070), 128.11 MiB | 12.58 MiB/s, done.
Resolving deltas: 100% (381/381), done.


In [79]:
%cd sam2

/content/sam2


In [80]:
!pip install -e . -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for SAM-2 (pyproject.toml) ... done


In [82]:
%cd checkpoints

/content/sam2/checkpoints


In [83]:
!./download_ckpts.sh

--2026-03-03 18:25:31--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.8.76.89, 65.8.76.77, 65.8.76.47, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.8.76.89|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 156008466 (149M) [application/vnd.snesdev-page-table]
Saving to: ‘sam2.1_hiera_tiny.pt’

sam2.1_hiera_tiny.p 100%[===================>] 148.78M   203MB/s    in 0.7s    

2026-03-03 18:25:32 (203 MB/s) - ‘sam2.1_hiera_tiny.pt’ saved [156008466/156008466]

--2026-03-03 18:25:32--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.8.76.89, 65.8.76.77, 65.8.76.47, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.8.76.89|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 184416285 (176M) [application/vnd.snesdev-

In [84]:
%cd ..

/content/sam2


In [85]:
!cp sam2/configs/sam2.1/*.yaml sam2

### Install Segment Anything Model 2 CPP Wrapper

In [86]:
!git clone https://github.com/ryouchinsa/sam-cpp-macos.git

Cloning into 'sam-cpp-macos'...
remote: Enumerating objects: 355, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 355 (delta 69), reused 24 (delta 24), pack-reused 282 (from 2)
Receiving objects: 100% (355/355), 815.95 KiB | 4.34 MiB/s, done.
Resolving deltas: 100% (220/220), done.


In [87]:
!cp sam-cpp-macos/export_onnx.py .
!cp sam-cpp-macos/david-tomaseti-Vw2HZQ1FGjU-unsplash.jpg .

### Export an ONNX model and check how the ONNX model works

In [88]:
!pip install torch==2.8.0 torchvision --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128


In [89]:
!pip install opencv-python onnx optimum[onnxruntime-gpu] onnxsim matplotlib numba

In [90]:
!python export_onnx.py --mode export

sam2.1_hiera_t.yaml
checkpoints/sam2.1_hiera_tiny.pt
/content/sam2/export_onnx.py:136: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/content/sam2/sam2/modeling/backbones/utils.py:30: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other input

In [91]:
!python export_onnx.py --mode import

['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
2026-03-03 18:28:02.169251851 [W:onnxruntime:, graph.cc:122 MergeShapeInfo] Error merging shape info for output. '/conv_s0/Conv_output_0' source:{1,32,256,256} target:{}. Falling back to lenient merge.
2026-03-03 18:28:02.169889765 [W:onnxruntime:, graph.cc:122 MergeShapeInfo] Error merging shape info for output. '/conv_s1/Conv_output_0' source:{1,64,128,128} target:{}. Falling back to lenient merge.
[1, 3, 1024, 1024]
['input']
['image_embeddings', 'high_res_features1', 'high_res_features2']
encoder start
infer time: 917.32 ms
['image_embeddings', 'high_res_features1', 'high_res_features2', 'point_coords', 'point_labels', 'mask_input', 'has_mask_input', 'orig_im_size']
['masks', 'iou_predictions', 'low_res_masks']
decoder start
infer time: 242.58 ms
decoder start
infer time: 157.09 ms
[ WARN:0@4.690] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
decod

In [92]:
!cp -r checkpoints/sam2.1_tiny sam-cpp-macos

In [93]:
%cd sam-cpp-macos

/content/sam2/sam-cpp-macos


### Build and run

In [94]:
!cmake -S . -B build -DONNXRUNTIME_ROOT_DIR=/content/onnxruntime-linux-x64-gpu-1.23.2

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found OpenCV: /usr (found version "4.5.4")
-- Configuring done (0.6s)
-- Generating done (0.0s)
-- Build files have been written to: /content/sam2/sam-cpp-macos/build


In [95]:
!cmake --build build

[ 20%] Building CXX object CMakeFiles/sam_cpp_lib.dir/sam.cpp.o
[ 40%] Building CXX object CMakeFiles/sam_cpp_lib.dir/util.cpp.o
[ 60%] Linking CXX shared library libsam_cpp_lib.so
[ 60%] Built target sam_cpp_lib
[ 80%] Building CXX object CMakeFiles/sam_cpp_test.dir/test.cpp.o
[100%] Linking CXX executable sam_cpp_test
[100%] Built target sam_cpp_test


In [97]:
!./build/sam_cpp_test -encoder="sam2.1_tiny/sam2.1_tiny_preprocess.onnx" -decoder="sam2.1_tiny/sam2.1_tiny.onnx" -image="david-tomaseti-Vw2HZQ1FGjU-unsplash.jpg" -device="cuda:0"

loadModel started
2026-03-03 18:28:55.329220462 [W:onnxruntime:, graph.cc:120 MergeShapeInfo] Error merging shape info for output. '/conv_s0/Conv_output_0' source:{1,32,256,256} target:{}. Falling back to lenient merge.
2026-03-03 18:28:55.330139452 [W:onnxruntime:, graph.cc:120 MergeShapeInfo] Error merging shape info for output. '/conv_s1/Conv_output_0' source:{1,64,128,128} target:{}. Falling back to lenient merge.
2026-03-03 18:28:55.705243874 [W:onnxruntime:, transformer_memcpy.cc:111 ApplyImpl] 10 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-03-03 18:28:55.709399913 [W:onnxruntime:, session_state.cc:1316 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly a